In [ ]:
#!pip install torch-geometric
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import HypergraphConv

# Step 1, load the structural connectome
# here synthetic data  (example for 10 nodes)
adj_matrix = torch.tensor([
    [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],  # Node 0 is connected to Node 1
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],  # Nodes 0, 1, 2 form a clique
    [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],  # Nodes 1, 2, 3 form a clique
    [0, 0, 1, 1, 1, 0, 0, 0, 0, 0],  # Nodes 2, 3, 4 form a clique
    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],  # Node 3 is connected to Node 4
    [0, 0, 0, 0, 0, 1, 1, 1, 0, 0],  # Nodes 5, 6, 7 form a clique
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 0],  # Node 6 is connected to Node 5 and 7
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 0],  # Nodes 6, 7, 8 form a clique
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],  # Nodes 7, 8, 9 form a clique
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],  # Node 9 is connected to Node 8
], dtype=torch.float)

# Step 2: Extract hyperedges (cliques) from the adjacency matrix
# In this example, we assume cliques are predefined, is this correct?:
hyperedges = [
    [0, 1, 2],  # Clique 1
    [1, 2, 3],  # Clique 2
    [2, 3, 4],  # Clique 3
    [5, 6, 7],  # Clique 4
    [6, 7, 8],  # Clique 5
    [7, 8, 9],  # Clique 6
]

# Step 3: Convert cliques into edge_index format
num_nodes = adj_matrix.size(0)
num_hyperedges = len(hyperedges)
row = torch.tensor([node for clique in hyperedges for node in clique])  # Nodes
col = torch.arange(num_hyperedges).repeat_interleave(torch.tensor([len(clique) for clique in hyperedges])) + num_nodes
 # Virtual hyperedge nodes
edge_index = torch.stack([row, col], dim=0)  # Shape [2, num_edges]

# Step 4: Define features and prepare the data
num_features = 2  # Node feature dimension
x = torch.rand((num_nodes, num_features))  # True values?
x_input = torch.cat([torch.rand((num_nodes, num_features)), torch.zeros((num_hyperedges, num_features))], dim=0)  # Include virtual nodes
data = Data(x=x_input, edge_index=edge_index, y=x)

# Step 5: Define and train the model
class HypergraphNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(HypergraphNet, self).__init__()
        self.conv1 = HypergraphConv(in_channels, hidden_channels)
        self.conv2 = HypergraphConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# Model setup
in_channels = x_input.size(1)
hidden_channels = 16
out_channels = x.size(1)
model = HypergraphNet(in_channels, hidden_channels, out_channels)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.mse_loss(out[:num_nodes], data.y)  # Exclude virtual nodes
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data.x, data.edge_index)
    mse = F.mse_loss(out[:num_nodes], data.y).item()
    return mse

for epoch in range(1, 101):
    loss = train()
    mse = test()
    if epoch % 10 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Test MSE: {mse:.4f}")

# Final predictions
model.eval()
predicted_features = model(data.x, data.edge_index)[:num_nodes]
print("Predicted features at nodes:")
print(predicted_features)


x.size(1):2
Epoch: 010, Loss: 0.0650, Test MSE: 0.0582
Epoch: 020, Loss: 0.0474, Test MSE: 0.0486
Epoch: 030, Loss: 0.0487, Test MSE: 0.0479
Epoch: 040, Loss: 0.0443, Test MSE: 0.0444
Epoch: 050, Loss: 0.0448, Test MSE: 0.0448
Epoch: 060, Loss: 0.0443, Test MSE: 0.0442
Epoch: 070, Loss: 0.0442, Test MSE: 0.0442
Epoch: 080, Loss: 0.0442, Test MSE: 0.0442
Epoch: 090, Loss: 0.0442, Test MSE: 0.0442
Epoch: 100, Loss: 0.0441, Test MSE: 0.0441
Predicted features at nodes:
tensor([[0.2749, 0.4495],
        [0.2775, 0.4563],
        [0.2846, 0.4681],
        [0.2894, 0.4775],
        [0.2987, 0.4919],
        [0.3051, 0.5197],
        [0.3024, 0.5137],
        [0.2986, 0.5049],
        [0.2953, 0.4975],
        [0.2909, 0.4873]], grad_fn=<SliceBackward0>)
